# Public portfolio version - Silver layer

This notebook is a sanitized public portfolio version focused on the Silver layer of the SenseTemp project. It shows how raw telemetry is validated, normalized, and enriched with sensor metadata before being promoted to curated outputs.

# Silver layer overview

The Silver layer is where data quality and enrichment are applied. Raw telemetry is transformed into a trustworthy curated dataset that supports operational analysis.

## What happens here

* validate mandatory fields and timestamps
* detect invalid placeholder values such as `999`
* reject sensors that do not exist in the metadata catalog
* normalize telemetry into a structure ready for joins
* enrich each reading with threshold and ownership metadata

## Why this matters

This layer turns raw measurements into reliable, contextualized observations. It is the foundation for alert logic and Gold reporting.

In [0]:
# Public portfolio configuration example
storage_account = "<your-storage-account>"

path_telemetria_json = f"wasbs://bronze@{storage_account}.blob.core.windows.net/telemetria/sample/*/*/*/*/*.json"
path_metadata_csv = f"wasbs://silver@{storage_account}.blob.core.windows.net/metadata/sample/sensores_limites.csv"
path_errores = f"wasbs://silver@{storage_account}.blob.core.windows.net/errores_delta/"
path_telemetria_validada = f"wasbs://silver@{storage_account}.blob.core.windows.net/telemetria_validada_delta/"
path_metadata_delta = f"wasbs://silver@{storage_account}.blob.core.windows.net/metadata_sensores_delta/"

print("Silver portfolio notebook loaded.")
print("Replace demo paths and configure secure access before execution.")

In [0]:
# Apply data quality rules and build curated Silver outputs
from pyspark.sql.functions import (
    array,
    col,
    concat_ws,
    current_timestamp,
    expr,
    lit,
    size,
    to_timestamp,
    when,
)
from pyspark.sql.types import ArrayType, StringType

print("Reading telemetry and metadata from demo paths...")
df_telemetria = spark.read.json(path_telemetria_json)
df_metadata = spark.read.option("header", "true").csv(path_metadata_csv)

sensor_cols = [c for c in df_telemetria.columns if c.startswith("Temp_")]
metadata_sensor_ids = [row["SensorID"] for row in df_metadata.select("SensorID").distinct().collect()]

sensor_cols_with_metadata = [c for c in sensor_cols if c in metadata_sensor_ids]
sensor_cols_without_metadata = [c for c in sensor_cols if c not in metadata_sensor_ids]
base_cols = [c for c in df_telemetria.columns if not c.startswith("Temp_")]

if not sensor_cols_with_metadata:
    raise ValueError("No sensor columns with valid metadata were found.")

sensor_invalid_array_raw = array(*[
    when(col(c) == lit(999), lit(f"sensor_valor_999:{c}"))
    for c in sensor_cols_with_metadata
]) if sensor_cols_with_metadata else lit([]).cast(ArrayType(StringType()))

sensor_without_metadata_array_raw = array(*[
    when(col(c).isNotNull(), lit(f"sensor_sin_metadata:{c}"))
    for c in sensor_cols_without_metadata
]) if sensor_cols_without_metadata else lit([]).cast(ArrayType(StringType()))

df_telemetria_quality = (
    df_telemetria
    .withColumn("FechaHoraTs", to_timestamp("FechaHoraRecepcion"))
    .withColumn("error_fecha_hora", when(col("FechaHoraRecepcion").isNull() | col("FechaHoraTs").isNull(), lit("fecha_hora_invalida")))
    .withColumn("error_nombre_dispositivo", when(col("NombreDispositivo").isNull(), lit("nombre_dispositivo_nulo")))
    .withColumn("sensor_invalid_array_raw", sensor_invalid_array_raw)
    .withColumn("sensor_without_metadata_array_raw", sensor_without_metadata_array_raw)
    .withColumn("sensor_invalid_array", expr("filter(sensor_invalid_array_raw, x -> x is not null)"))
    .withColumn("sensor_without_metadata_array", expr("filter(sensor_without_metadata_array_raw, x -> x is not null)"))
    .withColumn(
        "ReglasIncumplidas",
        concat_ws(
            " | ",
            col("error_fecha_hora"),
            col("error_nombre_dispositivo"),
            concat_ws(" | ", col("sensor_invalid_array")),
            concat_ws(" | ", col("sensor_without_metadata_array")),
        ),
    )
    .withColumn(
        "EsValido",
        (col("error_fecha_hora").isNull())
        & (col("error_nombre_dispositivo").isNull())
        & (size(col("sensor_invalid_array")) == 0)
        & (size(col("sensor_without_metadata_array")) == 0),
    )
)

df_telemetria_errores = (
    df_telemetria_quality
    .filter(~col("EsValido"))
    .drop("sensor_invalid_array_raw", "sensor_without_metadata_array_raw")
    .withColumn("FechaProceso", current_timestamp())
)

df_telemetria_clean = (
    df_telemetria_quality
    .filter(col("EsValido"))
    .select(*base_cols, *sensor_cols_with_metadata)
)

df_metadata_delta = (
    df_metadata
    .select(
        col("SensorID"),
        col("Ubicacion"),
        col("TempMin").cast("double").alias("TempMin"),
        col("TempMax").cast("double").alias("TempMax"),
        col("Responsable"),
    )
)

(
    df_telemetria_errores.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(path_errores)
)
(
    df_telemetria_clean.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(path_telemetria_validada)
)
(
    df_metadata_delta.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(path_metadata_delta)
)

display(df_telemetria_clean)
display(df_telemetria_errores)
display(df_metadata_delta)

In [0]:
# Normalize telemetry to sensor-level events and enrich with metadata
from pyspark.sql.functions import expr

sensor_cols = [c for c in df_telemetria_clean.columns if c.startswith("Temp_")]
if not sensor_cols:
    raise ValueError("No valid sensors with metadata are available for transformation.")

stack_expr = ", ".join([f"'{c}', `{c}`" for c in sensor_cols])

df_telemetria_long = df_telemetria_clean.select(
    "FechaHoraRecepcion",
    "NombreDispositivo",
    expr(f"stack({len(sensor_cols)}, {stack_expr}) as (SensorID, Temperatura)"),
)

df_joined = df_telemetria_long.join(df_metadata_delta, on="SensorID", how="inner")
display(df_joined)

# What to show in GitHub

For a public repository, this Silver notebook should demonstrate:

* concrete data quality rules
* rejected records versus valid curated records
* metadata-driven enrichment
* normalized sensor-level observations

Recommended GitHub file name:

`notebooks/02_silver_quality_enrichment.ipynb`